# Week 7 Lab: Logistic Regression from the Inside

**Core** (Exercises 1–2, about 35 minutes): the logistic function, and a logistic-regression classifier in which you write the few lines that *are* the algorithm. Everything else, including the sklearn plumbing, is written for you.

**Extension** (Exercises 3–4, optional): softmax, and logistic regression for more than two classes.

The math you need is stated in each exercise. If you're curious where the formulas come from, the full derivation is optional reading in `lectures/07-classification-logistic-regression/extra/3-gradient-descent-cross-entropy.ipynb`.

#### **Exercise 1**

The logistic (sigmoid) function turns any number $z$ into a probability between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

1. Implement `logistic_function(z)`. It should work on a whole NumPy array at once (use `np.exp`).
2. Run the cell. The plot shows $\sigma(z + \alpha)$ for three values of $\alpha$ (the intercept). In one sentence: what does changing $\alpha$ do to the curve?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def logistic_function(z):
    # YOUR CODE HERE
    pass


x = np.linspace(-10, 10, 200)
alphas = [-5, 0, 5]  # different intercepts shift the curve

plt.figure(figsize=(10, 5))
for alpha in alphas:
    plt.plot(x, logistic_function(x + alpha), label=f'alpha = {alpha}')
plt.axhline(0.5, color='gray', linestyle=':')
plt.title('The logistic function, shifted by the intercept')
plt.xlabel('x')
plt.ylabel('Probability')
plt.legend()
plt.grid(True)
plt.show()

_Answer here_

#### **Exercise 2**

Logistic regression predicts a probability with the logistic function applied to a linear combination of the features:

$$p = \sigma(Xw + b)$$

It is trained with **gradient descent**, exactly like the one you wrote in week 6. The gradient of the log loss turns out to be remarkably simple:

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^T (p - y) \qquad \frac{\partial L}{\partial b} = \frac{1}{n} \sum (p - y)$$

In words: **(prediction error) × (feature), averaged over the samples.** It's the same shape as the linear-regression gradient; the only difference is that the prediction $p$ now goes through the sigmoid.

The class below is complete except for the lines marked `# YOUR CODE HERE`:
1. `_predict_proba`: compute $p = \sigma(Xw + b)$ (use `np.dot`, `self.coef_`, `self.intercept_`, and your `logistic_function`).
2. In `fit`: compute the two gradients `dw` and `db`, then update `self.coef_` and `self.intercept_` (step *against* the gradient, scaled by `self.learning_rate`).

Then run the test cell, which trains your classifier and scikit-learn's on the same data.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array

class MyLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, learning_rate=0.1, n_iterations=1000, tolerance=1e-6):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.tolerance = tolerance

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape

        # Start with all weights at zero
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0.0

        for _ in range(self.n_iterations):
            p = self._predict_proba(X)

            # Gradients of the log loss
            dw = None  # YOUR CODE HERE
            db = None  # YOUR CODE HERE

            # Gradient descent step
            # YOUR CODE HERE: update self.coef_ and self.intercept_

            # Stop early once the updates become tiny
            if np.all(np.abs(self.learning_rate * dw) < self.tolerance):
                break
        return self

    def _predict_proba(self, X):
        # YOUR CODE HERE: return the probability of class 1 for each row of X
        pass

    def predict_proba(self, X):
        return self._predict_proba(check_array(X))

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
# TEST: breast-cancer diagnosis (malignant vs. benign) from 30 tumor measurements.
# Because your class follows the sklearn API, it drops straight into a Pipeline.
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

mine = make_pipeline(StandardScaler(), MyLogisticRegression()).fit(X_train, y_train)
theirs = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)

print(f"My test accuracy:      {accuracy_score(y_test, mine.predict(X_test)):.3f}")
print(f"sklearn test accuracy: {accuracy_score(y_test, theirs.predict(X_test)):.3f}")
print(f"Predictions that agree: {(mine.predict(X_test) == theirs.predict(X_test)).mean():.1%}")

**Questions**

1. How close is your classifier to scikit-learn's? Why might the two not agree perfectly? (Hint: sklearn uses a different optimizer, and a penalty on large weights by default.)
2. Try `MyLogisticRegression(learning_rate=0.001)` and `MyLogisticRegression(n_iterations=10)`. What happens, and why?

_Answer here_

---

### Extension exercises (optional)

> Exercises 3–4 extend logistic regression to **more than two classes**. The lecture introduced the idea (softmax, the "Handling multiple classes" section); here you build it. Skip these if you're short on time.

#### **Exercise 3** (extension)

**Softmax** generalizes the sigmoid to $K$ classes. It turns a row of $K$ scores $z_1, \dots, z_K$ into $K$ probabilities that sum to 1:

$$p_j = \frac{e^{z_j}}{\sum_{k=1}^K e^{z_k}}$$

Implement `softmax(z)` for a 2-D array where **each row** is one sample. (Tip: `np.sum(..., axis=1, keepdims=True)` sums across each row and keeps the shape so the division works. Optional: subtract each row's max from `z` first. It doesn't change the answer but prevents overflow.)

In [ ]:
def softmax(z):
    # YOUR CODE HERE
    pass


example_z = np.array([[1.0, 2.0, 0.5], [-1.0, 0.0, 3.0]])
example_softmax = softmax(example_z)
print("Input:")
print(example_z)
print("Softmax output:")
print(example_softmax.round(3))
print("Sum of probabilities for each row:", np.sum(example_softmax, axis=1))

#### **Exercise 4** (extension)

Now multiclass logistic regression. Each class gets its own row of weights, so `coef_` has shape `(n_classes, n_features)`. The labels are one-hot encoded as `y_onehot`, and the gradient has **the same form as Exercise 2**, just with matrices:

$$\frac{\partial L}{\partial W} = \frac{1}{n} (P - Y)^T X \qquad \frac{\partial L}{\partial b} = \frac{1}{n} \sum_{\text{rows}} (P - Y)$$

where $P$ are the softmax probabilities and $Y$ is `y_onehot`. Fill in the marked lines, then run the test on the iris data (3 species).

In [ ]:
from sklearn.preprocessing import LabelEncoder

class MyMulticlassLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, learning_rate=0.1, n_iterations=1000, tolerance=1e-6):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.tolerance = tolerance

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        self.classes_ = self.label_encoder_.classes_
        n_classes = len(self.classes_)
        n_samples, n_features = X.shape

        self.coef_ = np.zeros((n_classes, n_features))
        self.intercept_ = np.zeros(n_classes)
        y_onehot = np.eye(n_classes)[y_encoded]  # e.g. class 2 of 3 -> [0, 0, 1]

        for _ in range(self.n_iterations):
            P = self._predict_proba(X)
            dW = None  # YOUR CODE HERE, shape (n_classes, n_features)
            db = None  # YOUR CODE HERE, shape (n_classes,)
            # YOUR CODE HERE: update self.coef_ and self.intercept_

            if np.all(np.abs(self.learning_rate * dW) < self.tolerance):
                break
        return self

    def _predict_proba(self, X):
        # YOUR CODE HERE: scores = X W^T + b, then softmax
        pass

    def predict_proba(self, X):
        return self._predict_proba(check_array(X))

    def predict(self, X):
        return self.label_encoder_.inverse_transform(np.argmax(self.predict_proba(X), axis=1))


# TEST: iris, 3 species
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

mine = make_pipeline(StandardScaler(), MyMulticlassLogisticRegression()).fit(X_train, y_train)
theirs = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
print(f"My test accuracy:      {accuracy_score(y_test, mine.predict(X_test)):.3f}")
print(f"sklearn test accuracy: {accuracy_score(y_test, theirs.predict(X_test)):.3f}")